# SASHIMI-W standard API walkthrough

Use the installed package from any working directory. The candidate package
requires the locally supplied `sashimi-itamae` wheel until release publication.
The power convention is an explicit scientific choice: q5 uses the historical
published power suppression; q10 squares the Viel transfer amplitude. This
example selects q5, with no default change implied.

The small grid demonstrates the API and is not a converged abundance prediction.

In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import sashimi_w
from sashimi_w import Subhalos, PUBLISHED_Q5
from itamae.types import WeightedSubhaloCatalog
from itamae.provenance import source_revision

print("W source:", source_revision("sashimi-w", module_file=sashimi_w.__file__))
model = Subhalos(mass_wdm=2.0, wdm_power_convention=PUBLISHED_Q5)
parameters = dict(M0=1e10, redshift=0., dz=.5, zmax=1., N_ma=4,
                  N_herm=2, N_hermNa=3, logmamin=6., logmamax=8.)
catalog = model.rs_rhos_catalog_calc(**parameters)
assert catalog.shape == (16,)
print(catalog.metadata["calculation_specification"])
print("Expected surviving subhalos on this grid:", catalog.weight_final.sum())

The named catalog uses physical solar masses, megaparsecs and solar masses
per cubic megaparsec. `weight_base`, `weight_concentration` and
`weight_survival` remain separate. The final weight is their product.

In [ ]:
for name in ("m200_acc", "m_bound", "r_s", "rho_s", "c_t"):
    assert np.isfinite(catalog.columns[name]).all()
assert (catalog.weight_final >= 0).all()
assert (catalog.columns["m_bound"] <= catalog.columns["m200_acc"]).all()
assert np.array_equal(catalog.columns["survive"], catalog.columns["c_t"] > .77)
print("Units:", catalog.metadata["canonical_units"])
print("Weight factors:", sorted(catalog.weights))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(catalog.columns["m_bound"], catalog.weight_final,
           c=catalog.columns["z_acc"], cmap="viridis", s=35)
ax.set(xscale="log", yscale="log", xlabel="Bound mass [solar masses]",
       ylabel="Expected count per quadrature node", title="WDM example catalog: q5, 2 keV")
fig.tight_layout()
plt.show()

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "wdm-catalog.npz"
    catalog.to_npz(path)
    restored = WeightedSubhaloCatalog.from_npz(path)
    np.testing.assert_array_equal(restored.weight_final, catalog.weight_final)
    assert dict(restored.metadata) == dict(catalog.metadata)
print("Serialization preserved columns, independent weights and provenance.")

`rs_rhos_calc` converts the same calculation to a ten-element tuple
(Msun, kpc, Msun/pc³); tuple weight excludes survival. `subhalo_distr`, `N_sat`
and `N_sat_Vthres` remain available. Their historical counting definitions
are under a separate scientific review and are not used for the count above.

The separate scientific-validation notebook compares independent frozen
references and checks the continuous variance derivative. No legacy runtime
or `physics_mode` parameter remains in the standard API.